# Checkpointed 06b sub-01 baseline and matched tri-modal comparison

This notebook first reproduces historical 06b route_c for **sub-01, seed 0** from the full training partition. It performs CPU-only input validation and a CPU update profile before the separately gated GPU run. It never loads official-test EEG, test image features, or the merged train/test InternViT cache. After the valid baseline completes, an optional matched A/B/C/D warm-start run evaluates direct EEG-to-caption retrieval using captions from training images only. No official-test scores are produced.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os, pathlib, subprocess, sys
BRANCH='research/trimodal-baseline-transfer-20261120'
REPO=pathlib.Path('/content/eeg-alignment-phase1-06c')
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
else:
    branch=subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    if branch!=BRANCH or dirty: raise RuntimeError('Existing notebook checkout is on another branch or has local changes; preserve it and use a fresh runtime/path.')
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
actual=subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
assert actual==BRANCH, f'Expected {BRANCH}, got {actual}'
sys.path.insert(0,str(REPO))
import numpy as np, torch
print('branch:',actual,'commit:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
print('PyTorch:',torch.__version__,'CUDA available:',torch.cuda.is_available())

## Required full historical training cache

Default cache root: `/content/drive/MyDrive/eeg_results/cache`. Required: `sub-01_train_z16.npy` with shape `[16540,63,time]`, plus `feat6_internvit6b_parts/train_000.pt` through `train_008.pt`. Each train chunk must contain mean-pooled layers 20/24/28/32/36, width 3200, aligned in the original 06b training image order. The CPU preflight rejects any missing, changed, nonfinite, wrong-shape, or wrong-protocol input. It never falls back to the 2,000-image pilot or opens test files. See `docs/HISTORICAL_06B_CHECKPOINTED_REPRODUCTION.md` for exact requirements and the optional train-only caption artifact contract.

In [ ]:
from phase1.historical_06b import create_manifest, preflight, profile_cpu
CACHE=pathlib.Path('/content/drive/MyDrive/eeg_results/cache')
RUN_ROOT=pathlib.Path('/content/drive/MyDrive/eeg_results/v6b_checkpointed_phase1/sub01_seed0_routec')
RUN_ROOT.mkdir(parents=True,exist_ok=True)
MANIFEST=RUN_ROOT/'train_only_manifest.json'
if not MANIFEST.exists(): create_manifest(CACHE,MANIFEST)
preflight_report=preflight(MANIFEST)
print(json.dumps(preflight_report,indent=2))
(RUN_ROOT/'cpu_preflight.json').write_text(json.dumps(preflight_report,indent=2))

In [ ]:
# CPU-only representative update; estimates the full 900-step schedule before GPU use.
profile=profile_cpu(MANIFEST,batch_size=32)
print(json.dumps(profile,indent=2))
(RUN_ROOT/'cpu_profile.json').write_text(json.dumps(profile,indent=2))
assert profile['official_test_opened'] is False

## Historical 06b reproduction

The code uses the historical posterior-17 channel order, train-only `z16` EEG unchanged, mean-pooled InternViT layers 20/24/28/32/36, 06b target normalization, original Aligner architecture, route logits `[-2,-1,0,-1,-2]`, AdamW and OneCycleLR (60 epochs, batch 1,024). It selects the best checkpoint using only the historical last-image-per-concept training validation rows. The model state is saved as a plain 06b `Aligner.state_dict()`; epoch-boundary checkpoints support `--resume`.

In [ ]:
from phase1.historical_06b import run_baseline
RUN_BASELINE=False  # Set True only after reviewing the CPU preflight/profile above.
if RUN_BASELINE:
    if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU only after CPU preflight passes.')
    baseline_result=run_baseline(MANIFEST,RUN_ROOT/'baseline','cuda',resume=True)
    print(json.dumps({k:v for k,v in baseline_result.items() if k not in ('target_mean','target_std','fit_rows','validation_rows')},indent=2))
else:
    print('GPU training is gated. Set RUN_BASELINE=True after reviewing CPU preflight/profile.')

## Matched tri-modal A/B/C/D (only after the baseline)

This stage refuses to run unless the checkpointed 06b baseline is complete and verified. It additionally requires a separate **training-only** 16,540 × 512 caption embedding array, aligned exact-caption IDs, and a provenance JSON documenting caption generator/encoder/revision, training-only normalization, and `used_test: false`. The mixed legacy caption cache is never opened. All four arms warm-start from identical baseline weights and new-head initialization, use the same historical fit/validation split and training budget, and report image and duplicate-aware text retrieval. Direct text scores use EEG and caption candidates only. If those train-only caption inputs are unavailable, this optional stage skips with no fallback.

In [ ]:
from phase1.trimodal_06b import preflight_text_inputs, run as run_trimodal
TEXT_FEATURES=CACHE/'captions_train_only.npy'
CAPTION_IDS=CACHE/'caption_ids_train_only.json'
CAPTION_PROVENANCE=CACHE/'caption_provenance_train_only.json'
caption_files=[TEXT_FEATURES,CAPTION_IDS,CAPTION_PROVENANCE]
if all(path.is_file() for path in caption_files):
    print(json.dumps(preflight_text_inputs(*caption_files),indent=2))
else:
    print('Tri-modal stage not ready; missing train-only caption inputs:',[str(path) for path in caption_files if not path.is_file()])

In [ ]:
RUN_TRIMODAL=False  # Set True only after baseline completion and train-only caption preflight.
if RUN_TRIMODAL:
    if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU only after both CPU preflights pass.')
    if not all(path.is_file() for path in caption_files): raise FileNotFoundError('Train-only caption artifacts are required; mixed caches are not accepted.')
    records=run_trimodal(MANIFEST,RUN_ROOT/'baseline',TEXT_FEATURES,CAPTION_IDS,CAPTION_PROVENANCE,RUN_ROOT/'trimodal_abcd','cuda',resume=True)
    print(json.dumps(records,indent=2))
else:
    print('A/B/C/D GPU training is gated. Set RUN_TRIMODAL=True after baseline and caption preflights pass.')

## Scientific boundary

The historical test evaluation is intentionally absent. Baseline selection and tri-modal reporting use training-partition validation only. The original 06b result and this development validation are not an untouched-test comparison. This notebook does not claim SOTA, significance, or a tri-modal gain. Preserve each `run.json`, input hash report, CPU profile, epoch log, and checkpoint with the Drive output.